# SDXL 免費生圖（Google Colab T4）

這份筆記本可以在 **Google Colab 免費版** 直接跑，不用綁卡、不用付費。

## 使用前設定

1. 點上方選單 **執行階段 → 變更執行階段類型**
2. 硬體加速器選 **T4 GPU**（免費版提供，顯存 16GB）
3. 按儲存後，再從第一格開始依序執行

## 第一次跑會花多久？

- 第一次會下載 SDXL 模型，約 **7.11GB**
- 模型載入約 **62 秒**
- 生圖設定：**1024x1024、30 步**，一張約 **53 秒**

## 跑完記得關 GPU

生完圖請到 **執行階段 → 中斷連線並刪除執行階段**，把 GPU 關掉，就不占額度。


In [ ]:
# 檢查 GPU：確認有抓到 T4、顯存約 16GB
!nvidia-smi

In [ ]:
# 安裝需要的套件
!pip install -q diffusers transformers accelerate safetensors

## ⚠️ 提醒：Hugging Face 授權

SDXL 官方模型 `stabilityai/stable-diffusion-xl-base-1.0` 要先在 Hugging Face 接受授權才能下載。

如果下一格下載時報 **401** 錯誤，請這樣做：

1. 到 Hugging Face 打開 SDXL 模型頁，登入後點接受授權（Agree）
2. 到 Hugging Face 的 Settings → Access Tokens，產生一個免費的 **Read（讀取）** token
3. 回到下一格，把這兩行的註解拿掉，並把 token 貼上：

```python
# from huggingface_hub import login
# login("在這裡貼你的 HF token")
```

改成取消註解、貼上你自己的 token 後，再重新執行下一格即可。**不要把 token 分享給別人。**

---

**備用方案（免授權模型）：** 如果不想處理授權，可以改用免授權的模型。把下一格載入模型的那一行改成下面這行（已註解，取消註解並取代原行即可）：

```python
# pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V4.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
```


In [ ]:
# 載入 SDXL 模型（第一次會下載約 7.11GB，請耐心等）
import torch
import time
from diffusers import StableDiffusionXLPipeline

# 如果下載報 401，先到 Hugging Face 接受 SDXL 授權、產生免費讀取 token，
# 再取消下面兩行的註解、貼上你的 token：
# from huggingface_hub import login
# login("在這裡貼你的 HF token")

# 備用免授權模型（如需使用，取消註解下一行，並註解掉再下一行的官方模型載入）：
# pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V4.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)

start = time.time()

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)
pipe.to("cuda")
pipe.enable_attention_slicing()

print(f"模型載入完成，用時 {time.time() - start:.1f} 秒")

In [ ]:
# 生成第一張圖：橘貓坐在陽光窗台上
# 提示詞用英文效果比較好，改 prompt 字串就能換圖
import time
from IPython.display import display

prompt = "An orange cat sitting on a sunny windowsill, green plants beside it, warm sunlight, photorealistic, highly detailed"
negative_prompt = "blurry, low quality, distorted, deformed"

start = time.time()

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=1024,
    width=1024,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]

print(f"生成完成，用時 {time.time() - start:.1f} 秒")

# 存檔
image.save("/content/orange_cat.png")
print("已存到 /content/orange_cat.png")

# 顯示圖片
display(image)

In [ ]:
# 再生成一張（示範換提示詞）：日出山湖
# 一樣只要改 prompt 就能換主題
import time
from IPython.display import display

prompt = "A mountain lake at sunrise, mist, pine trees, photorealistic landscape"
negative_prompt = "blurry, low quality, distorted, deformed"

start = time.time()

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=1024,
    width=1024,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]

print(f"生成完成，用時 {time.time() - start:.1f} 秒")

image.save("/content/lake.png")
print("已存到 /content/lake.png")

display(image)

## ✅ 收尾提醒

生完圖記得關 GPU，才不占額度：

> **執行階段 → 中斷連線並刪除執行階段**

圖片都存在 `/content/` 裡：

- `/content/orange_cat.png`（第一張：橘貓）
- `/content/lake.png`（第二張：山湖）

在 Colab 左邊的 **檔案區** 就能找到這兩個檔案，點右邊的 ⋮ 選 **下載** 即可存到自己電腦。

注意：中斷連線後 `/content/` 會被清空，要下載請在關 GPU 之前先下載。
